# K4: train a sequence model

Generated by `scripts/make_notebooks.py`; see `docs/kaggle_runbook.md`. Run it headless:
**Save Version -> Save & Run All (Commit)**. No cell needs editing.

- **Accelerator:** **GPU T4 x2**. **Internet:** ON.
- **Attach (Add Input):** the K2 output for SIZE; to resume or continue, also the output of the earlier K4 version(s) of the same CONFIG (detected automatically)
- **Expected time:** 50m ~0.5-1 h per config, 500m up to 8 h per session (then resume)
- **Output (`/kaggle/working/out`):** `results/<name>-{val,test}-*.json`, `ckpt/<name>/{val,test}/`, markers `_HOOKREC_RESULTS_<name>`, `_HOOKREC_CKPT`

If the log ends with **RESUME NEEDED**, the version still succeeded: save a new version with this version's output attached and it continues from the checkpoint.

In [ ]:
# Parameters: the defaults work unedited (attach inputs, then Save Version ->
# Save & Run All). Generated by scripts/make_notebooks.py.
REPO_URL = "https://github.com/Arubad/Reccomendation_system.git"  # the HookSeq repo (public: no token needed; GITHUB_TOKEN only for a private fork)
GIT_REF = "v0.3"  # branch or tag to run (pinned to the release tag of the reported runs)
SIZE = "50m"  # Yambda size: '50m' or '500m'
CONFIG = "configs/ablations/a0_sasrec.yaml"  # run name = file stem
RUN_SPLITS = ["val", "test"]  # val (early stopping), then the test refit
OVERRIDES = []  # e.g. ["train.lr=3e-4"] (key=value, YAML values)
RESUME_INPUTS = True  # resume from / carry over attached earlier K4 outputs
REQUIRE_GPU = True  # fail fast if the GPU accelerator is not on

# fixed Kaggle paths
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"

import os
import time

# train.max_hours is measured from here (the session start), not from the script start
os.environ.setdefault("HOOKREC_SESSION_START", str(int(time.time())))
print("HOOKREC_SESSION_START =", os.environ["HOOKREC_SESSION_START"])

In [ ]:
# Clone the repo and install it. The image's torch is used (no torch install).
import importlib
import os
import shutil
import subprocess
import sys


import base64
import re
import threading
from importlib import metadata
from urllib.parse import urlsplit


def _stream(cmd, timeout):
    """Run cmd, streaming its output; raise on a non-zero exit (a `!cmd` magic would not)."""
    p = subprocess.Popen(
        cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    expired = []
    timer = threading.Timer(timeout, lambda: (expired.append(1), p.kill()))
    timer.start()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
    finally:
        timer.cancel()
        p.stdout.close()
    code = p.wait()
    if expired:
        raise RuntimeError(f"{' '.join(cmd[:5])} ... timed out after {timeout} s (network?)")
    if code != 0:
        raise RuntimeError(f"{' '.join(cmd[:5])} ... exited {code}")


if os.path.exists(REPO):
    shutil.rmtree(REPO)
token = basic = u = ""
try:  # optional, only for a private fork: Add-ons -> Secrets -> GITHUB_TOKEN (attached)
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN") or ""
except Exception:
    token = ""
# never prompt (a headless run would hang); the token travels only in this process's env
# as an HTTP header for REPO_URL's host: never in argv, the URL, .git/config or .git/logs
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "never"}
if token:
    u = urlsplit(REPO_URL)
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0=f"http.{u.scheme}://{u.netloc}/.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}",
    )
try:
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, REPO],
        env=git_env, stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=600,
    )
except subprocess.TimeoutExpired:
    raise RuntimeError(f"git clone {REPO_URL} -b {GIT_REF} timed out after 600 s") from None
if r.returncode != 0:
    log = r.stdout + r.stderr
    if token:
        log = log.replace(token, "***").replace(basic, "***")
    raise RuntimeError(
        f"git clone {REPO_URL} -b {GIT_REF} failed (exit {r.returncode}):\n{log}\n"
        "Check REPO_URL / GIT_REF and that Internet is ON; a private fork needs the "
        "GITHUB_TOKEN secret attached to this notebook."
    )
del token, git_env, r, basic, u
_stream([sys.executable, "-m", "pip", "install", "-q", "-e", REPO], timeout=1800)
sys.path.insert(0, os.path.join(REPO, "src"))
importlib.invalidate_caches()
# the installed (not an already-imported, possibly stale) polars must meet pyproject's floor
pl_version = metadata.version("polars")
if tuple(int(x) for x in re.findall(r"\d+", pl_version)[:2]) < (2, 1):
    raise RuntimeError(
        f"polars {pl_version} is installed but hookrec needs polars>=2.1 (pl.Float16, "
        "streaming collect); pip did not upgrade it. Check the pip log above."
    )
SHA = subprocess.run(
    ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print(f"hookrec {REPO_URL} {GIT_REF} @ {SHA} (polars {pl_version})")

In [ ]:
# Environment + input discovery (inputs are found by marker files; no slugs to edit).
import os
import sys
from pathlib import Path

from hookrec.utils.kaggle import (
    check_disk, env_report, find_inputs, find_one, find_processed, has_run, run_logged,
)

env_report(WORK)

import torch

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError(
        "no GPU: set Accelerator = GPU T4 x2 in the notebook sidebar (Settings), then Save "
        "Version again. (REQUIRE_GPU = False trains on CPU, which is far too slow for Yambda.)"
    )
PROCESSED = find_processed(SIZE, INPUT_ROOT)
NAME = Path(CONFIG).stem
import yaml

from hookrec.train.trainer import training_hash
from hookrec.utils.config import load_config

# the training config this version runs (as run_train.py parses OVERRIDES): a carried-over
# record of another config (another hash, or none) never counts as a finished split
_over = {k: yaml.safe_load(v) for k, _, v in (o.partition("=") for o in OVERRIDES)}
THASH = training_hash(load_config(os.path.join(REPO, CONFIG), _over))
# earlier K4 outputs of this config: their results are carried over, their checkpoints resumed
PREV_RESULTS = find_inputs(f"_HOOKREC_RESULTS_{NAME}", INPUT_ROOT) if RESUME_INPUTS else []
PREV_CKPTS = [d for d in find_inputs("_HOOKREC_CKPT", INPUT_ROOT) if d.name == NAME]
PREV_CKPTS = PREV_CKPTS if RESUME_INPUTS else []
print("processed:", PROCESSED)
print("run:", NAME, "| earlier results:", PREV_RESULTS, "| earlier checkpoints:", PREV_CKPTS)

In [ ]:
import shutil
import sys

RESULTS = Path(OUT, "results")
CKPT = Path(OUT, "ckpt", NAME)
RESULTS.mkdir(parents=True, exist_ok=True)
CKPT.mkdir(parents=True, exist_ok=True)
(RESULTS / f"_HOOKREC_RESULTS_{NAME}").write_text("")
(CKPT / "_HOOKREC_CKPT").write_text("")
for d in PREV_RESULTS:
    for p in sorted(d.glob("*.json")):
        if not (RESULTS / p.name).exists():
            shutil.copyfile(p, RESULTS / p.name)
            print("carried over", p.name)
# a finished (carried-over) split brings its weights along, so the newest K4 output alone is
# enough for K5 (test/model_final.pt); outputs that hold the split's record are preferred
for split in ("val", "test"):
    rec = f"{NAME}-{split}"
    if not has_run(RESULTS, rec, PROCESSED.name, THASH):
        continue
    homes = sorted(
        PREV_CKPTS,
        key=lambda d: not has_run(d.parent.parent / "results", rec, PROCESSED.name, THASH),
    )
    for fname in ("model_final.pt", "best.pt"):
        dest = CKPT / split / fname
        src = next((d / split / fname for d in homes if (d / split / fname).exists()), None)
        if src is not None and not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(src, dest)
            print("carried over", dest.relative_to(OUT))

RESUME_NEEDED = False
for split in RUN_SPLITS:  # test takes its step count from the val record, so val runs first
    run_name = f"{NAME}-{split}"
    if has_run(RESULTS, run_name, PROCESSED.name, THASH):
        print(f"{run_name}: a record already exists, skipping")
        continue
    cmd = [
        sys.executable, f"{REPO}/scripts/run_train.py", "--processed", PROCESSED,
        "--config", os.path.join(REPO, CONFIG), "--split", split, "--ckpt", CKPT / split,
        "--out", RESULTS,
    ]
    for d in PREV_CKPTS:
        if (d / split).is_dir():
            cmd += ["--resume", d / split]
    for o in OVERRIDES:
        cmd += ["--override", o]
    run_logged(cmd, cwd=REPO)
    # exit 0 without a record = max_hours
    if not has_run(RESULTS, run_name, PROCESSED.name, THASH):
        RESUME_NEEDED = True
        BANNER = (
            "=" * 78 + f"\nRESUME NEEDED: {run_name} stopped on train.max_hours; its checkpoint is "
            f"in out/ckpt/{NAME}/{split}.\nThis version is fine. Save a NEW version of this "
            "notebook with THIS version's output attached\n(Add Input -> Your Work -> this "
            "notebook): it resumes automatically.\n" + "=" * 78
        )
        print(BANNER)
        break
    # the record exists: step checkpoints (resume state, ~2.3 GB each at 500M) are not needed
    # any more; best.pt / model_final.pt stay (K5 loads model_final.pt)
    for p in sorted((CKPT / split).glob("ckpt-*")):
        p.unlink()
    print(f"{run_name}: done; removed its step checkpoints")

In [ ]:
from hookrec.eval.report import results_table
from hookrec.utils.results import load_runs

check_disk(WORK)
for p in sorted(CKPT.rglob("*.pt")):
    print(f"{p.stat().st_size / 1e9:8.3f} GB  {p.relative_to(OUT)}")
print(results_table(load_runs(RESULTS), split="test", size=SIZE))
print(BANNER if RESUME_NEEDED else f"{NAME}: {', '.join(RUN_SPLITS)} done")